In [3]:
!pip install -q transformers datasets scikit-learn numpy

import json, torch, os, zipfile
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from torch.utils.data import Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments
from sklearn.metrics import f1_score, classification_report
from IPython.display import FileLink

# 1. LOAD DATA
TRAIN_PATH = "/kaggle/input/datasets/alfeysani/psydefetect/train.json"
TEST_PATH = "/kaggle/input/datasets/alfeysani/psydefetect/test.json"

with open(TRAIN_PATH) as f: train_data = json.load(f)
with open(TEST_PATH) as f: test_data = json.load(f)

# 2. SPLIT & GET RAW WEIGHTS
labels = [d['label'] for d in train_data]
train_split, val_split = train_test_split(train_data, test_size=0.1, random_state=42, stratify=labels)

raw_weights = compute_class_weight('balanced', classes=np.arange(9), y=[d['label'] for d in train_split])
class_weights_tensor = torch.tensor(raw_weights, dtype=torch.float)
print("⚖️ Using RAW Class Weights to force learning of rare classes:\n", np.round(raw_weights, 2))

# 3. DATASET PREP
def format_input(example, k=10):
    history = example['dialogue'][:-1]
    last_k = history[-k:] if len(history) >= k else history
    context_text = " [SEP] ".join([f"{t['speaker']}: {t['text']}" for t in last_k])
    return f"{context_text} [SEP] TARGET: {example['current_text']}"

class DefenseDataset(Dataset):
    def __init__(self, data, tokenizer, max_len=512, is_test=False):
        self.data, self.tokenizer, self.max_len, self.is_test = data, tokenizer, max_len, is_test
    def __len__(self): return len(self.data)
    def __getitem__(self, idx):
        encoding = self.tokenizer(format_input(self.data[idx]), max_length=self.max_len, padding='max_length', truncation=True, return_tensors='pt')
        result = {'input_ids': encoding['input_ids'].squeeze(), 'attention_mask': encoding['attention_mask'].squeeze()}
        if not self.is_test: result['labels'] = torch.tensor(self.data[idx]['label'], dtype=torch.long)
        return result

model_name = "mental/mental-roberta-base"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=9)

train_ds = DefenseDataset(train_split, tokenizer)
val_ds   = DefenseDataset(val_split, tokenizer)
test_ds  = DefenseDataset(test_data, tokenizer, is_test=True)

# 4. CUSTOM TRAINER (Multi-GPU Safe)
class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop('labels')
        outputs = model(**inputs)
        loss_fct = torch.nn.CrossEntropyLoss(weight=class_weights_tensor.to(labels.device))
        loss = loss_fct(outputs.logits, labels)
        return (loss, outputs) if return_outputs else loss

def compute_metrics(eval_pred):
    preds = np.argmax(eval_pred.predictions, axis=-1)
    return {'f1_macro': f1_score(eval_pred.label_ids, preds, average='macro')}

training_args = TrainingArguments(
    output_dir='/kaggle/working/roberta_final', num_train_epochs=10, 
    per_device_train_batch_size=8, gradient_accumulation_steps=2, 
    per_device_eval_batch_size=32, learning_rate=1e-5, weight_decay=0.01,
    eval_strategy='epoch', save_strategy='epoch', load_best_model_at_end=True,
    metric_for_best_model='f1_macro', save_total_limit=1, save_only_model=True, fp16=True, report_to='none'
)

trainer = WeightedTrainer(model=model, args=training_args, train_dataset=train_ds, eval_dataset=val_ds, compute_metrics=compute_metrics)

# 5. TRAIN & EVALUATE
print("\n🚀 Training MentalRoBERTa (Champion Run)...")
trainer.train()

print("\n🎯 Final Validation Evaluation:")
val_preds = trainer.predict(val_ds)
final_val_preds = np.argmax(val_preds.predictions, axis=-1)
golds = [d['label'] for d in val_split]
print(classification_report(golds, final_val_preds))
print("🔥 FINAL VAL MACRO F1:", f1_score(golds, final_val_preds, average='macro'))

# 6. GENERATE SUBMISSION
print("\n📦 Generating CodaBench Submission...")
test_preds = np.argmax(trainer.predict(test_ds).predictions, axis=-1)
for i, entry in enumerate(test_data): entry['label'] = int(test_preds[i])

with open('/kaggle/working/prediction.json', 'w') as f: json.dump(test_data, f)
with zipfile.ZipFile('/kaggle/working/submission.zip', 'w') as z: z.write('/kaggle/working/prediction.json', arcname='prediction.json')

print("✅ DONE! Download your submission.zip from the Kaggle Output menu on the right.")

⚖️ Using RAW Class Weights to force learning of rare classes:
 [0.7  1.92 3.39 2.09 2.45 4.33 1.2  0.21 7.45]


config.json:   0%|          | 0.00/682 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/327 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['roberta.embeddings.LayerNorm.weight', 'roberta.embeddings.LayerNorm.bias', 'roberta.encoder.layer.0.attention.output.LayerNorm.weight', 'roberta.encoder.layer.0.attention.output.LayerNorm.bias', 'roberta.encoder.layer.0.output.LayerNorm.weight', 'roberta.encoder.layer.0.output.LayerNorm.bias', 'roberta.encoder.layer.1.attention.output.LayerNorm.weight', 'roberta.encoder.layer.1.attention.output.LayerNorm.bias', 'roberta.encoder.layer.1.output.LayerNorm.weight', 'roberta.encoder.layer.1.output.LayerNorm.bias', 'roberta.encoder.layer.2.attention.output.LayerNorm.weight', 'roberta.encoder.layer.2.attention.output.LayerNorm.bias', 'roberta.encoder.layer.2.output.LayerNorm.weight', 'roberta.encoder.layer.2.output.LayerNorm.bias', 'roberta.encoder.layer.3.attention.output.LayerNorm.weight', 'roberta.encoder.layer.3.attention.output.LayerNorm.bias', 'roberta.encoder.layer.3.output.LayerNorm.weight', 'roberta.encoder.layer.3.output.Laye


🎯 Final Validation Evaluation:


              precision    recall  f1-score   support

           0       0.72      0.97      0.83        30
           1       0.27      0.64      0.38        11
           2       0.22      0.33      0.27         6
           3       0.12      0.20      0.15        10
           4       0.00      0.00      0.00         8
           5       0.00      0.00      0.00         5
           6       0.25      0.35      0.29        17
           7       0.77      0.48      0.59        97
           8       0.00      0.00      0.00         3

    accuracy                           0.50       187
   macro avg       0.26      0.33      0.28       187
weighted avg       0.57      0.50      0.51       187

🔥 FINAL VAL MACRO F1: 0.2794535847947395

📦 Generating CodaBench Submission...


✅ DONE! Download your submission.zip from the Kaggle Output menu on the right.


In [2]:
!pip install -q huggingface_hub
from huggingface_hub import login
from kaggle_secrets import UserSecretsClient

try:
    user_secrets = UserSecretsClient()
    hf_token = user_secrets.get_secret("HF_TOKEN")
    login(token=hf_token)
    print("✅ Logged into Hugging Face! You now have access to gated models.")
except Exception as e:
    print("❌ Error: Could not find HF_TOKEN. Did you add it to Kaggle Secrets and turn the toggle on?")

✅ Logged into Hugging Face! You now have access to gated models.


In [5]:
!pip install -q transformers datasets scikit-learn numpy

import json, torch, zipfile
import numpy as np
from sklearn.model_selection import train_test_split
from torch.utils.data import Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments
from sklearn.metrics import f1_score, accuracy_score, classification_report

# 1. LOAD DATA
TRAIN_PATH = "/kaggle/input/datasets/alfeysani/psydefetect/train.json"
TEST_PATH = "/kaggle/input/datasets/alfeysani/psydefetect/test.json"

with open(TRAIN_PATH) as f: train_data = json.load(f)
with open(TEST_PATH) as f: test_data = json.load(f)

# 2. SPLIT DATA
labels = [d['label'] for d in train_data]
train_split, val_split = train_test_split(train_data, test_size=0.1, random_state=42, stratify=labels)

# 3. DATASET PREP
def format_input(example, k=10):
    history = example['dialogue'][:-1]
    last_k = history[-k:] if len(history) >= k else history
    context_text = " [SEP] ".join([f"{t['speaker']}: {t['text']}" for t in last_k])
    return f"{context_text} [SEP] TARGET: {example['current_text']}"

class DefenseDataset(Dataset):
    def __init__(self, data, tokenizer, max_len=512, is_test=False):
        self.data, self.tokenizer, self.max_len, self.is_test = data, tokenizer, max_len, is_test
    def __len__(self): return len(self.data)
    def __getitem__(self, idx):
        encoding = self.tokenizer(format_input(self.data[idx]), max_length=self.max_len, padding='max_length', truncation=True, return_tensors='pt')
        result = {'input_ids': encoding['input_ids'].squeeze(), 'attention_mask': encoding['attention_mask'].squeeze()}
        if not self.is_test: result['labels'] = torch.tensor(self.data[idx]['label'], dtype=torch.long)
        return result

model_name = "mental/mental-roberta-base"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=9)

train_ds = DefenseDataset(train_split, tokenizer)
val_ds   = DefenseDataset(val_split, tokenizer)
test_ds  = DefenseDataset(test_data, tokenizer, is_test=True)

# 4. STANDARD TRAINER (Optimized for ACCURACY)
def compute_metrics(eval_pred):
    preds = np.argmax(eval_pred.predictions, axis=-1)
    return {
        'accuracy': accuracy_score(eval_pred.label_ids, preds),
        'f1_macro': f1_score(eval_pred.label_ids, preds, average='macro')
    }

training_args = TrainingArguments(
    output_dir='/kaggle/working/roberta_acc_optimized', 
    num_train_epochs=8, 
    per_device_train_batch_size=8, 
    gradient_accumulation_steps=2, 
    per_device_eval_batch_size=32, 
    learning_rate=2e-5, 
    weight_decay=0.01,
    eval_strategy='epoch', 
    save_strategy='epoch', 
    load_best_model_at_end=True,
    metric_for_best_model='accuracy', # 🛠️ Changed to Accuracy
    save_total_limit=1, 
    save_only_model=True, 
    fp16=True, 
    report_to='none'
)

# Using standard Trainer, NO class weights
trainer = Trainer(model=model, args=training_args, train_dataset=train_ds, eval_dataset=val_ds, compute_metrics=compute_metrics)

# 5. TRAIN & EVALUATE
print("\n🚀 Training MentalRoBERTa (Accuracy Optimized)...")
trainer.train()

print("\n🎯 Final Validation Evaluation:")
val_preds = trainer.predict(val_ds)
final_val_preds = np.argmax(val_preds.predictions, axis=-1)
golds = [d['label'] for d in val_split]
print(classification_report(golds, final_val_preds))
print("🔥 FINAL VAL ACCURACY:", accuracy_score(golds, final_val_preds))

# 6. GENERATE SUBMISSION
print("\n📦 Generating CodaBench Submission...")
test_preds = np.argmax(trainer.predict(test_ds).predictions, axis=-1)
for i, entry in enumerate(test_data): entry['label'] = int(test_preds[i])

with open('/kaggle/working/prediction.json', 'w') as f: json.dump(test_data, f)
with zipfile.ZipFile('/kaggle/working/submission.zip', 'w') as z: z.write('/kaggle/working/prediction.json', arcname='prediction.json')

print("✅ DONE! Download your submission.zip from the Kaggle Output menu on the right.")

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: mental/mental-roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.dense.bias              | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
classifier.out_proj.weight      | MISSING    | 
classifier.out_proj.bias        | MISSING    | 
classifier.dense.bias           | MISSING    | 
classifier.dense.weight         | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.



🚀 Training MentalRoBERTa (Accuracy Optimized)...


/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Epoch,Training Loss,Validation Loss


KeyboardInterrupt: 

In [6]:
!pip install -q transformers datasets scikit-learn numpy

import json, torch, zipfile
import numpy as np
from sklearn.model_selection import train_test_split
from torch.utils.data import Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments
from sklearn.metrics import f1_score, accuracy_score, classification_report
from IPython.display import FileLink

# 1. LOAD DATA
TRAIN_PATH = "/kaggle/input/datasets/alfeysani/psydefetect/train.json"
TEST_PATH = "/kaggle/input/datasets/alfeysani/psydefetect/test.json"

with open(TRAIN_PATH) as f: train_data = json.load(f)
with open(TEST_PATH) as f: test_data = json.load(f)

# 2. SPLIT DATA (Seed 42 to keep it consistent with your 0.56 run)
labels = [d['label'] for d in train_data]
train_split, val_split = train_test_split(train_data, test_size=0.1, random_state=42, stratify=labels)

# 3. DATASET PREP
def format_input(example, k=10):
    history = example['dialogue'][:-1]
    last_k = history[-k:] if len(history) >= k else history
    context_text = " [SEP] ".join([f"{t['speaker']}: {t['text']}" for t in last_k])
    return f"{context_text} [SEP] TARGET: {example['current_text']}"

class DefenseDataset(Dataset):
    def __init__(self, data, tokenizer, max_len=512, is_test=False):
        self.data, self.tokenizer, self.max_len, self.is_test = data, tokenizer, max_len, is_test
    def __len__(self): return len(self.data)
    def __getitem__(self, idx):
        encoding = self.tokenizer(format_input(self.data[idx]), max_length=self.max_len, padding='max_length', truncation=True, return_tensors='pt')
        result = {'input_ids': encoding['input_ids'].squeeze(), 'attention_mask': encoding['attention_mask'].squeeze()}
        if not self.is_test: result['labels'] = torch.tensor(self.data[idx]['label'], dtype=torch.long)
        return result

# 🛠️ SWITCHED BACK TO MENTALBERT
model_name = "mental/mental-bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=9)

train_ds = DefenseDataset(train_split, tokenizer)
val_ds   = DefenseDataset(val_split, tokenizer)
test_ds  = DefenseDataset(test_data, tokenizer, is_test=True)

# 4. STANDARD TRAINER (Strictly optimizing for ACCURACY)
def compute_metrics(eval_pred):
    preds = np.argmax(eval_pred.predictions, axis=-1)
    return {
        'accuracy': accuracy_score(eval_pred.label_ids, preds),
        'f1_macro': f1_score(eval_pred.label_ids, preds, average='macro')
    }

training_args = TrainingArguments(
    output_dir='/kaggle/working/bert_acc_optimized', 
    num_train_epochs=8, 
    per_device_train_batch_size=8, 
    gradient_accumulation_steps=2, 
    per_device_eval_batch_size=32, 
    learning_rate=2e-5, 
    weight_decay=0.01,
    eval_strategy='epoch', 
    save_strategy='epoch', 
    load_best_model_at_end=True,
    metric_for_best_model='accuracy', # 🎯 The Trainer will now pick the best Accuracy epoch
    save_total_limit=1, 
    save_only_model=True, 
    fp16=True, 
    report_to='none'
)

# Standard Trainer, NO custom loss, NO class weights
trainer = Trainer(model=model, args=training_args, train_dataset=train_ds, eval_dataset=val_ds, compute_metrics=compute_metrics)

# 5. TRAIN & EVALUATE
print("\n🚀 Training MentalBERT (Accuracy Optimized)...")
trainer.train()

print("\n🎯 Final Validation Evaluation:")
val_preds = trainer.predict(val_ds)
final_val_preds = np.argmax(val_preds.predictions, axis=-1)
golds = [d['label'] for d in val_split]
print(classification_report(golds, final_val_preds, zero_division=0))
print("🔥 FINAL VAL ACCURACY:", accuracy_score(golds, final_val_preds))

# 6. GENERATE SUBMISSION
print("\n📦 Generating CodaBench Submission...")
test_preds = np.argmax(trainer.predict(test_ds).predictions, axis=-1)
for i, entry in enumerate(test_data): entry['label'] = int(test_preds[i])

with open('/kaggle/working/prediction.json', 'w') as f: json.dump(test_data, f)
with zipfile.ZipFile('/kaggle/working/submission.zip', 'w') as z: z.write('/kaggle/working/prediction.json', arcname='prediction.json')

print("✅ DONE! Download your submission.zip from the Kaggle Output menu on the right.")

config.json:   0%|          | 0.00/639 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/321 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

...

There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.attention.output.La


🎯 Final Validation Evaluation:


/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


              precision    recall  f1-score   support

           0       0.82      0.90      0.86        30
           1       0.20      0.09      0.12        11
           2       0.00      0.00      0.00         6
           3       0.00      0.00      0.00        10
           4       0.00      0.00      0.00         8
           5       0.00      0.00      0.00         5
           6       0.50      0.06      0.11        17
           7       0.63      0.95      0.76        97
           8       0.00      0.00      0.00         3

    accuracy                           0.65       187
   macro avg       0.24      0.22      0.20       187
weighted avg       0.52      0.65      0.55       187

🔥 FINAL VAL ACCURACY: 0.6470588235294118

📦 Generating CodaBench Submission...


✅ DONE! Download your submission.zip from the Kaggle Output menu on the right.


In [7]:
!pip install -q transformers datasets scikit-learn numpy

import json, torch, zipfile
import numpy as np
import torch.nn.functional as F
from torch import nn
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from torch.utils.data import Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments
from sklearn.metrics import f1_score, classification_report

# 1. LOAD DATA
TRAIN_PATH = "/kaggle/input/datasets/alfeysani/psydefetect/train.json"
TEST_PATH = "/kaggle/input/datasets/alfeysani/psydefetect/test.json"

with open(TRAIN_PATH) as f: train_data = json.load(f)
with open(TEST_PATH) as f: test_data = json.load(f)

# 2. SPLIT & GET RAW WEIGHTS
labels = [d['label'] for d in train_data]
train_split, val_split = train_test_split(train_data, test_size=0.1, random_state=42, stratify=labels)

raw_weights = compute_class_weight('balanced', classes=np.arange(9), y=[d['label'] for d in train_split])
class_weights_tensor = torch.tensor(raw_weights, dtype=torch.float)
print("⚖️ Raw Class Weights:\n", np.round(raw_weights, 2))

# 3. DATASET PREP
def format_input(example, k=10):
    history = example['dialogue'][:-1]
    last_k = history[-k:] if len(history) >= k else history
    context_text = " [SEP] ".join([f"{t['speaker']}: {t['text']}" for t in last_k])
    return f"{context_text} [SEP] TARGET: {example['current_text']}"

class DefenseDataset(Dataset):
    def __init__(self, data, tokenizer, max_len=512, is_test=False):
        self.data, self.tokenizer, self.max_len, self.is_test = data, tokenizer, max_len, is_test
    def __len__(self): return len(self.data)
    def __getitem__(self, idx):
        encoding = self.tokenizer(format_input(self.data[idx]), max_length=self.max_len, padding='max_length', truncation=True, return_tensors='pt')
        result = {'input_ids': encoding['input_ids'].squeeze(), 'attention_mask': encoding['attention_mask'].squeeze()}
        if not self.is_test: result['labels'] = torch.tensor(self.data[idx]['label'], dtype=torch.long)
        return result

model_name = "mental/mental-roberta-base"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=9)

train_ds = DefenseDataset(train_split, tokenizer)
val_ds   = DefenseDataset(val_split, tokenizer)
test_ds  = DefenseDataset(test_data, tokenizer, is_test=True)

# 4. 🛠️ THE FOCAL LOSS IMPLEMENTATION
class FocalLoss(nn.Module):
    def __init__(self, weight=None, gamma=2.0, reduction='mean'):
        super(FocalLoss, self).__init__()
        self.weight = weight # We pass our rare-class weights here
        self.gamma = gamma   # The focusing parameter
        self.reduction = reduction

    def forward(self, inputs, targets):
        ce_loss = F.cross_entropy(inputs, targets, weight=self.weight, reduction='none')
        pt = torch.exp(-ce_loss)
        focal_loss = ((1 - pt) ** self.gamma) * ce_loss
        if self.reduction == 'mean': return focal_loss.mean()
        return focal_loss.sum()

class FocalTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop('labels')
        outputs = model(**inputs)
        # Apply Focal Loss to force the model to learn the hard examples
        loss_fct = FocalLoss(weight=class_weights_tensor.to(labels.device), gamma=2.0)
        loss = loss_fct(outputs.logits, labels)
        return (loss, outputs) if return_outputs else loss

def compute_metrics(eval_pred):
    preds = np.argmax(eval_pred.predictions, axis=-1)
    return {'f1_macro': f1_score(eval_pred.label_ids, preds, average='macro')}

# 5. TRAINER CONFIG
training_args = TrainingArguments(
    output_dir='/kaggle/working/roberta_focal', num_train_epochs=10, 
    per_device_train_batch_size=8, gradient_accumulation_steps=2, 
    per_device_eval_batch_size=32, learning_rate=1e-5, weight_decay=0.01,
    eval_strategy='epoch', save_strategy='epoch', load_best_model_at_end=True,
    metric_for_best_model='f1_macro', save_total_limit=1, save_only_model=True, fp16=True, report_to='none'
)

trainer = FocalTrainer(model=model, args=training_args, train_dataset=train_ds, eval_dataset=val_ds, compute_metrics=compute_metrics)

# 6. TRAIN & EVALUATE
print("\n🚀 Training MentalRoBERTa with FOCAL LOSS...")
trainer.train()

print("\n🎯 Final Validation Evaluation (F1 Focused):")
val_preds = trainer.predict(val_ds)
final_val_preds = np.argmax(val_preds.predictions, axis=-1)
golds = [d['label'] for d in val_split]
print(classification_report(golds, final_val_preds, zero_division=0))
print("🔥 FINAL VAL MACRO F1:", f1_score(golds, final_val_preds, average='macro'))

# 7. GENERATE SUBMISSION
print("\n📦 Generating CodaBench Submission...")
test_preds = np.argmax(trainer.predict(test_ds).predictions, axis=-1)
for i, entry in enumerate(test_data): entry['label'] = int(test_preds[i])

with open('/kaggle/working/prediction.json', 'w') as f: json.dump(test_data, f)
with zipfile.ZipFile('/kaggle/working/submission.zip', 'w') as z: z.write('/kaggle/working/prediction.json', arcname='prediction.json')

print("✅ DONE! Download your submission.zip.")

⚖️ Raw Class Weights:
 [0.7  1.92 3.39 2.09 2.45 4.33 1.2  0.21 7.45]


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: mental/mental-roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.dense.bias              | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
classifier.out_proj.weight      | MISSING    | 
classifier.out_proj.bias        | MISSING    | 
classifier.dense.bias           | MISSING    | 
classifier.dense.weight         | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.



🚀 Training MentalRoBERTa with FOCAL LOSS...


Epoch,Training Loss,Validation Loss,F1 Macro
1,No log,1.837043,0.018519
2,No log,1.817582,0.059252
3,No log,1.727525,0.181396
4,No log,1.601047,0.200820
5,No log,1.568958,0.195198
6,No log,1.504215,0.159400
7,No log,1.468522,0.220745
8,No log,1.460831,0.223971
9,No log,1.444945,0.221502
10,3.040403,1.448216,0.201340


...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['roberta.embeddings.LayerNorm.weight', 'roberta.embeddings.LayerNorm.bias', 'roberta.encoder.layer.0.attention.output.LayerNorm.weight', 'roberta.encoder.layer.0.attention.output.LayerNorm.bias', 'roberta.encoder.layer.0.output.LayerNorm.weight', 'roberta.encoder.layer.0.output.LayerNorm.bias', 'roberta.encoder.layer.1.attention.output.LayerNorm.weight', 'roberta.encoder.layer.1.attention.output.LayerNorm.bias', 'roberta.encoder.layer.1.output.LayerNorm.weight', 'roberta.encoder.layer.1.output.LayerNorm.bias', 'roberta.encoder.layer.2.attention.output.LayerNorm.weight', 'roberta.encoder.layer.2.attention.output.LayerNorm.bias', 'roberta.encoder.layer.2.output.LayerNorm.weight', 'roberta.encoder.layer.2.output.LayerNorm.bias', 'roberta.encoder.layer.3.attention.output.LayerNorm.weight', 'roberta.encoder.layer.3.attention.output.LayerNorm.bias', 'roberta.encoder.layer.3.output.LayerNorm.weight', 'roberta.encoder.layer.3.output.Laye


🎯 Final Validation Evaluation (F1 Focused):


              precision    recall  f1-score   support

           0       0.60      0.93      0.73        30
           1       0.25      0.27      0.26        11
           2       0.30      0.50      0.38         6
           3       0.10      0.10      0.10        10
           4       0.17      0.12      0.14         8
           5       0.09      0.60      0.15         5
           6       0.17      0.59      0.26        17
           7       0.00      0.00      0.00        97
           8       0.00      0.00      0.00         3

    accuracy                           0.26       187
   macro avg       0.18      0.35      0.22       187
weighted avg       0.15      0.26      0.18       187

🔥 FINAL VAL MACRO F1: 0.2239710772319468

📦 Generating CodaBench Submission...


✅ DONE! Download your submission.zip.


In [4]:
!pip install -q huggingface_hub transformers datasets scikit-learn numpy

from huggingface_hub import login
from kaggle_secrets import UserSecretsClient
import json, torch, zipfile, gc
import numpy as np
from sklearn.model_selection import train_test_split
from torch.utils.data import Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments
from sklearn.metrics import f1_score, accuracy_score

# Authenticate
user_secrets = UserSecretsClient()
hf_token = user_secrets.get_secret("HF_TOKEN")
login(token=hf_token)

# Data paths
TRAIN_PATH = "/kaggle/input/datasets/alfeysani/psydefetect/train.json"
TEST_PATH = "/kaggle/input/datasets/alfeysani/psydefetect/test.json"

with open(TRAIN_PATH) as f: train_data = json.load(f)
with open(TEST_PATH) as f: test_data = json.load(f)

labels = [d['label'] for d in train_data]
train_split, val_split = train_test_split(train_data, test_size=0.1, random_state=42, stratify=labels)

def format_input(example, k=10):
    history = example['dialogue'][:-1]
    last_k = history[-k:] if len(history) >= k else history
    context_text = " [SEP] ".join([f"{t['speaker']}: {t['text']}" for t in last_k])
    return f"{context_text} [SEP] TARGET: {example['current_text']}"

class DefenseDataset(Dataset):
    def __init__(self, data, tokenizer, max_len=512, is_test=False):
        self.data, self.tokenizer, self.max_len, self.is_test = data, tokenizer, max_len, is_test
    def __len__(self): return len(self.data)
    def __getitem__(self, idx):
        encoding = self.tokenizer(format_input(self.data[idx]), max_length=self.max_len, padding='max_length', truncation=True, return_tensors='pt')
        result = {'input_ids': encoding['input_ids'].squeeze(), 'attention_mask': encoding['attention_mask'].squeeze()}
        if not self.is_test: result['labels'] = torch.tensor(self.data[idx]['label'], dtype=torch.long)
        return result

In [5]:
model_name = "mental/mental-bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=9)

test_ds = DefenseDataset(test_data, tokenizer, is_test=True)

training_args = TrainingArguments(
    output_dir='/kaggle/working/bert_acc', num_train_epochs=8, 
    per_device_train_batch_size=8, gradient_accumulation_steps=2, 
    learning_rate=2e-5, eval_strategy='epoch', save_strategy='epoch', 
    load_best_model_at_end=True, metric_for_best_model='accuracy', 
    save_total_limit=1, save_only_model=True, fp16=True, report_to='none'
)

def compute_metrics(eval_pred):
    preds = np.argmax(eval_pred.predictions, axis=-1)
    return {'accuracy': accuracy_score(eval_pred.label_ids, preds)}

trainer = Trainer(model=model, args=training_args, train_dataset=DefenseDataset(train_split, tokenizer), 
                  eval_dataset=DefenseDataset(val_split, tokenizer), compute_metrics=compute_metrics)

trainer.train()
bert_test_logits = trainer.predict(test_ds).predictions
np.save("bert_acc_logits.npy", bert_test_logits)

# Clean memory
del model, trainer
torch.cuda.empty_cache()
gc.collect()

config.json:   0%|          | 0.00/639 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/321 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.attention.output.La

1126

In [6]:
from sklearn.utils.class_weight import compute_class_weight
raw_weights = compute_class_weight('balanced', classes=np.arange(9), y=[d['label'] for d in train_split])
weights_tensor = torch.tensor(raw_weights, dtype=torch.float)

model_name = "mental/mental-roberta-base"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=9)

class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop('labels')
        outputs = model(**inputs)
        loss_fct = torch.nn.CrossEntropyLoss(weight=weights_tensor.to(labels.device))
        return (loss_fct(outputs.logits, labels), outputs) if return_outputs else loss_fct(outputs.logits, labels)

training_args.output_dir = '/kaggle/working/rob_f1'
training_args.metric_for_best_model = 'f1_macro'

def compute_f1(eval_pred):
    preds = np.argmax(eval_pred.predictions, axis=-1)
    return {'f1_macro': f1_score(eval_pred.label_ids, preds, average='macro')}

trainer = WeightedTrainer(model=model, args=training_args, train_dataset=DefenseDataset(train_split, tokenizer), 
                          eval_dataset=DefenseDataset(val_split, tokenizer), compute_metrics=compute_f1)

trainer.train()
rob_test_logits = trainer.predict(DefenseDataset(test_data, tokenizer, is_test=True)).predictions
np.save("rob_f1_logits.npy", rob_test_logits)

config.json:   0%|          | 0.00/682 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/327 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['roberta.embeddings.LayerNorm.weight', 'roberta.embeddings.LayerNorm.bias', 'roberta.encoder.layer.0.attention.output.LayerNorm.weight', 'roberta.encoder.layer.0.attention.output.LayerNorm.bias', 'roberta.encoder.layer.0.output.LayerNorm.weight', 'roberta.encoder.layer.0.output.LayerNorm.bias', 'roberta.encoder.layer.1.attention.output.LayerNorm.weight', 'roberta.encoder.layer.1.attention.output.LayerNorm.bias', 'roberta.encoder.layer.1.output.LayerNorm.weight', 'roberta.encoder.layer.1.output.LayerNorm.bias', 'roberta.encoder.layer.2.attention.output.LayerNorm.weight', 'roberta.encoder.layer.2.attention.output.LayerNorm.bias', 'roberta.encoder.layer.2.output.LayerNorm.weight', 'roberta.encoder.layer.2.output.LayerNorm.bias', 'roberta.encoder.layer.3.attention.output.LayerNorm.weight', 'roberta.encoder.layer.3.attention.output.LayerNorm.bias', 'roberta.encoder.layer.3.output.LayerNorm.weight', 'roberta.encoder.layer.3.output.Laye

In [7]:
bert_logits = np.load("bert_acc_logits.npy")
rob_logits = np.load("rob_f1_logits.npy")

# Weighted average: We give BERT 40% weight for accuracy, RoBERTa 60% for F1
final_logits = (bert_logits * 0.4) + (rob_logits * 0.6)
final_preds = np.argmax(final_logits, axis=-1)

# Generate submission
for i, entry in enumerate(test_data):
    entry['label'] = int(final_preds[i])

with open('prediction.json', 'w') as f:
    json.dump(test_data, f)
with zipfile.ZipFile('submission.zip', 'w') as z:
    z.write('prediction.json')

print("✅ FINAL ENSEMBLE READY! Download submission.zip from the Output menu.")

✅ FINAL ENSEMBLE READY! Download submission.zip from the Output menu.


In [12]:
from sklearn.metrics import classification_report, accuracy_score, f1_score
import torch
import numpy as np
import gc
from transformers import AutoModelForSequenceClassification, Trainer

print("📊 Calculating Ensemble Scores on Validation...")

# 1. Get RoBERTa Val Logits 
# (Ensuring we use the dataset currently in your memory)
rob_val_logits = trainer.predict(val_ds).predictions

# 2. Reload MentalBERT to get its Val Logits
# Proper way to clear memory to avoid Out of Memory (OOM) errors
if 'bert_model_eval' in locals():
    del bert_model_eval
if 'bert_trainer_eval' in locals():
    del bert_trainer_eval

torch.cuda.empty_cache()
gc.collect()

print("🔄 Loading MentalBERT for validation check...")
bert_model_eval = AutoModelForSequenceClassification.from_pretrained("mental/mental-bert-base-uncased", num_labels=9).to("cuda")
bert_trainer_eval = Trainer(model=bert_model_eval)
bert_val_logits = bert_trainer_eval.predict(val_ds).predictions

# 3. Combine with your 0.4 / 0.6 weighting
val_final_logits = (bert_val_logits * 0.4) + (rob_val_logits * 0.6)
val_final_preds = np.argmax(val_final_logits, axis=-1)

# 4. Get ground truth labels
val_golds = [d['label'] for d in val_split]

print("\n--- 🏆 EXPECTED LEADERBOARD PERFORMANCE ---")
print(f"✅ Accuracy (Leaderboard UI): {accuracy_score(val_golds, val_final_preds):.4f}")
print(f"✅ Macro F1 (Winning Metric): {f1_score(val_golds, val_final_preds, average='macro'):.4f}")
print("\n" + classification_report(val_golds, val_final_preds, zero_division=0))

# Final Cleanup
del bert_model_eval, bert_trainer_eval
torch.cuda.empty_cache()
gc.collect()

📊 Calculating Ensemble Scores on Validation...


NameError: name 'val_ds' is not defined

In [13]:
!pip install -q huggingface_hub transformers datasets scikit-learn numpy

from huggingface_hub import login
from kaggle_secrets import UserSecretsClient
import json, torch, zipfile, gc, os
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from torch.utils.data import Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments
from sklearn.metrics import f1_score, accuracy_score, classification_report

# 1. SETUP & DATA
try:
    user_secrets = UserSecretsClient()
    login(token=user_secrets.get_secret("HF_TOKEN"))
except:
    print("HF Token not found, ensure Secrets are enabled.")

TRAIN_PATH = "/kaggle/input/datasets/alfeysani/psydefetect/train.json"
TEST_PATH = "/kaggle/input/datasets/alfeysani/psydefetect/test.json"

with open(TRAIN_PATH) as f: train_data = json.load(f)
with open(TEST_PATH) as f: test_data = json.load(f)

labels = [d['label'] for d in train_data]
train_split, val_split = train_test_split(train_data, test_size=0.1, random_state=42, stratify=labels)

def format_input(example, k=10):
    history = example['dialogue'][:-1]
    last_k = history[-k:] if len(history) >= k else history
    context_text = " [SEP] ".join([f"{t['speaker']}: {t['text']}" for t in last_k])
    return f"{context_text} [SEP] TARGET: {example['current_text']}"

class DefenseDataset(Dataset):
    def __init__(self, data, tokenizer, max_len=512, is_test=False):
        self.data, self.tokenizer, self.max_len, self.is_test = data, tokenizer, max_len, is_test
    def __len__(self): return len(self.data)
    def __getitem__(self, idx):
        encoding = self.tokenizer(format_input(self.data[idx]), max_length=self.max_len, padding='max_length', truncation=True, return_tensors='pt')
        result = {'input_ids': encoding['input_ids'].squeeze(), 'attention_mask': encoding['attention_mask'].squeeze()}
        if not self.is_test: result['labels'] = torch.tensor(self.data[idx]['label'], dtype=torch.long)
        return result

# 2. TRAIN MENTALBERT (ACCURACY POWERHOUSE)
print("--- Training MentalBERT (Accuracy) ---")
tok_bert = AutoTokenizer.from_pretrained("mental/mental-bert-base-uncased")
mod_bert = AutoModelForSequenceClassification.from_pretrained("mental/mental-bert-base-uncased", num_labels=9)

args_bert = TrainingArguments(
    output_dir='/kaggle/working/bert_acc', num_train_epochs=8, 
    per_device_train_batch_size=8, gradient_accumulation_steps=2, 
    learning_rate=2e-5, eval_strategy='epoch', save_strategy='epoch', 
    load_best_model_at_end=True, metric_for_best_model='accuracy', 
    save_total_limit=1, fp16=True, report_to='none'
)

trainer_bert = Trainer(
    model=mod_bert, args=args_bert, 
    train_dataset=DefenseDataset(train_split, tok_bert), 
    eval_dataset=DefenseDataset(val_split, tok_bert), 
    compute_metrics=lambda p: {'accuracy': accuracy_score(p.label_ids, np.argmax(p.predictions, axis=-1))}
)

trainer_bert.train()
bert_val_logits = trainer_bert.predict(DefenseDataset(val_split, tok_bert)).predictions
bert_test_logits = trainer_bert.predict(DefenseDataset(test_data, tok_bert, is_test=True)).predictions

# Clean memory for RoBERTa
del mod_bert, trainer_bert
torch.cuda.empty_cache()
gc.collect()

# 3. TRAIN MENTALROBERTA (F1 SPECIALIST)
print("\n--- Training MentalRoBERTa (F1) ---")
tok_rob = AutoTokenizer.from_pretrained("mental/mental-roberta-base")
mod_rob = AutoModelForSequenceClassification.from_pretrained("mental/mental-roberta-base", num_labels=9)

weights = torch.tensor(compute_class_weight('balanced', classes=np.arange(9), y=[d['label'] for d in train_split]), dtype=torch.float)

class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop('labels')
        logits = model(**inputs).logits
        loss = torch.nn.CrossEntropyLoss(weight=weights.to(logits.device))(logits, labels)
        return (loss, logits) if return_outputs else loss

args_rob = TrainingArguments(
    output_dir='/kaggle/working/rob_f1', num_train_epochs=8, 
    per_device_train_batch_size=8, gradient_accumulation_steps=2, 
    learning_rate=1e-5, eval_strategy='epoch', save_strategy='epoch', 
    load_best_model_at_end=True, metric_for_best_model='f1_macro', 
    save_total_limit=1, fp16=True, report_to='none'
)

trainer_rob = WeightedTrainer(
    model=mod_rob, args=args_rob, 
    train_dataset=DefenseDataset(train_split, tok_rob), 
    eval_dataset=DefenseDataset(val_split, tok_rob), 
    compute_metrics=lambda p: {'f1_macro': f1_score(p.label_ids, np.argmax(p.predictions, axis=-1), average='macro')}
)

trainer_rob.train()
rob_val_logits = trainer_rob.predict(DefenseDataset(val_split, tok_rob)).predictions
rob_test_logits = trainer_rob.predict(DefenseDataset(test_data, tok_rob, is_test=True)).predictions

# 4. ENSEMBLE & SCORES
print("\n--- CALCULATING ENSEMBLE VALIDATION SCORES ---")
# Combining: 40% BERT (Accuracy) + 60% RoBERTa (F1 Logic)
final_val_logits = (bert_val_logits * 0.4) + (rob_val_logits * 0.6)
final_val_preds = np.argmax(final_val_logits, axis=-1)
val_golds = [d['label'] for d in val_split]

print(f"🔥 Final Accuracy: {accuracy_score(val_golds, final_val_preds):.4f}")
print(f"🔥 Final Macro F1: {f1_score(val_golds, final_val_preds, average='macro'):.4f}")
print(classification_report(val_golds, final_val_preds, zero_division=0))

# 5. SUBMISSION
final_test_logits = (bert_test_logits * 0.4) + (rob_test_logits * 0.6)
final_test_preds = np.argmax(final_test_logits, axis=-1)

for i, entry in enumerate(test_data):
    entry['label'] = int(final_test_preds[i])

with open('prediction.json', 'w') as f: json.dump(test_data, f)
with zipfile.ZipFile('submission.zip', 'w') as z: z.write('prediction.json')

print("✅ ENSEMBLE COMPLETE. Download submission.zip from Output folder.")

--- Training MentalBERT (Accuracy) ---


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: mental/mental-bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
bert.embeddings.position_ids               | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
bert.pooler.dense.bias                     | MISSING    | 
classifier.weight                          | MISSING    | 
bert.pooler.dense.weight                   | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if 

Epoch,Training Loss,Validation Loss,Accuracy
1,No log,2.936924,0.540107
2,No log,2.733289,0.593583
3,No log,2.649636,0.614973
4,No log,2.357952,0.620321
5,No log,2.421257,0.631016
6,No log,2.377963,0.620321
7,No log,2.330197,0.620321
8,No log,2.423499,0.620321


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...


--- Training MentalRoBERTa (F1) ---


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: mental/mental-roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
classifier.out_proj.bias        | MISSING    | 
classifier.dense.bias           | MISSING    | 
classifier.out_proj.weight      | MISSING    | 
classifier.dense.weight         | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch,Training Loss,Validation Loss


ValueError: Found input variables with inconsistent numbers of samples: [187, 175]

In [15]:
from sklearn.metrics import f1_score, accuracy_score, classification_report
import numpy as np
import torch
import gc

# 1. CLEAN MEMORY FIRST
if 'trainer_rob' in locals(): del trainer_rob
if 'mod_rob' in locals(): del mod_rob
torch.cuda.empty_cache()
gc.collect()

# 2. ROBUST METRIC FUNCTION
def compute_metrics_final(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    # Ensure we only score where lengths match exactly
    min_len = min(len(labels), len(preds))
    return {
        'f1_macro': f1_score(labels[:min_len], preds[:min_len], average='macro'),
        'accuracy': accuracy_score(labels[:min_len], preds[:min_len])
    }

# 3. RE-START TRAINING
print("--- Starting Stable Training: MentalRoBERTa (F1) ---")
tok_rob = AutoTokenizer.from_pretrained("mental/mental-roberta-base")
mod_rob = AutoModelForSequenceClassification.from_pretrained("mental/mental-roberta-base", num_labels=9)

# Balanced weights to help F1 score
weights = torch.tensor(compute_class_weight('balanced', classes=np.arange(9), y=[d['label'] for d in train_split]), dtype=torch.float)

class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop('labels')
        logits = model(**inputs).logits
        loss = torch.nn.CrossEntropyLoss(weight=weights.to(logits.device))(logits, labels)
        return (loss, logits) if return_outputs else loss

args_rob = TrainingArguments(
    output_dir='/kaggle/working/rob_f1_stable', 
    num_train_epochs=8, 
    per_device_train_batch_size=8, 
    gradient_accumulation_steps=2, 
    per_device_eval_batch_size=8, # 🛠️ Reset to 8 to fix the dimension error
    learning_rate=1e-5, 
    eval_strategy='epoch', 
    save_strategy='epoch', 
    load_best_model_at_end=True, 
    metric_for_best_model='f1_macro', 
    save_total_limit=1, 
    fp16=True, 
    report_to='none'
)

trainer_rob = WeightedTrainer(
    model=mod_rob, args=args_rob, 
    train_dataset=DefenseDataset(train_split, tok_rob), 
    eval_dataset=DefenseDataset(val_split, tok_rob), 
    compute_metrics=compute_metrics_final 
)

trainer_rob.train()

# 4. GET FINAL LOGITS
print("--- Saving Final Logits ---")
rob_val_logits = trainer_rob.predict(DefenseDataset(val_split, tok_rob)).predictions
rob_test_logits = trainer_rob.predict(DefenseDataset(test_data, tok_rob, is_test=True)).predictions

# 5. GENERATE FINAL HYBRID SUBMISSION
# (Assuming bert_val_logits and bert_test_logits are still in memory from the BERT run)
print("\n--- CALCULATING HYBRID ENSEMBLE SCORES ---")
val_ensemble_logits = (bert_val_logits * 0.4) + (rob_val_logits * 0.6)
val_ensemble_preds = np.argmax(val_ensemble_logits, axis=-1)
val_golds = [d['label'] for d in val_split]

print(f"🔥 Hybrid Accuracy: {accuracy_score(val_golds, val_ensemble_preds):.4f}")
print(f"🔥 Hybrid Macro F1: {f1_score(val_golds, val_ensemble_preds, average='macro'):.4f}")

# FINAL ZIP
final_test_logits = (bert_test_logits * 0.4) + (rob_test_logits * 0.6)
final_test_preds = np.argmax(final_test_logits, axis=-1)
for i, entry in enumerate(test_data): entry['label'] = int(final_test_preds[i])
with open('prediction.json', 'w') as f: json.dump(test_data, f)
with zipfile.ZipFile('submission.zip', 'w') as z: z.write('prediction.json')

print("✅ SUCCESS! submission.zip is ready.")

--- Starting Stable Training: MentalRoBERTa (F1) ---


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: mental/mental-roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
classifier.out_proj.bias        | MISSING    | 
classifier.dense.bias           | MISSING    | 
classifier.out_proj.weight      | MISSING    | 
classifier.dense.weight         | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch,Training Loss,Validation Loss,F1 Macro,Accuracy
1,No log,2.182438,0.073668,0.331429
2,No log,2.103776,0.094973,0.342857
3,No log,2.070034,0.076597,0.228571
4,No log,1.864660,0.066935,0.188571
5,No log,1.870279,0.080691,0.200000
6,No log,1.805170,0.079047,0.211429
7,No log,1.837911,0.077354,0.234286
8,No log,1.861263,0.079998,0.234286


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['roberta.embeddings.LayerNorm.weight', 'roberta.embeddings.LayerNorm.bias', 'roberta.encoder.layer.0.attention.output.LayerNorm.weight', 'roberta.encoder.layer.0.attention.output.LayerNorm.bias', 'roberta.encoder.layer.0.output.LayerNorm.weight', 'roberta.encoder.layer.0.output.LayerNorm.bias', 'roberta.encoder.layer.1.attention.output.LayerNorm.weight', 'roberta.encoder.layer.1.attention.output.LayerNorm.bias', 'roberta.encoder.layer.1.output.LayerNorm.weight', 'roberta.encoder.layer.1.output.LayerNorm.bias', 'roberta.encoder.layer.2.attention.output.LayerNorm.weight', 'roberta.encoder.layer.2.attention.output.LayerNorm.bias', 'roberta.encoder.layer.2.output.LayerNorm.weight', 'roberta.encoder.layer.2.output.LayerNorm.bias', 'roberta.encoder.layer.3.attention.output.LayerNorm.weight', 'roberta.encoder.layer.3.attention.output.LayerNorm.bias', 'roberta.encoder.layer.3.output.LayerNorm.weight', 'roberta.encoder.layer.3.output.Laye

--- Saving Final Logits ---



--- CALCULATING HYBRID ENSEMBLE SCORES ---


ValueError: operands could not be broadcast together with shapes (187,9) (175,9) 

In [16]:
from sklearn.metrics import classification_report, accuracy_score, f1_score
import numpy as np
import torch
import gc

print("📊 Running Exhaustive Validation for Ensemble...")

# 1. Clear memory to start fresh
if 'trainer_rob' in locals(): del trainer_rob
if 'mod_rob' in locals(): del mod_rob
torch.cuda.empty_cache()
gc.collect()

# 2. Re-extract RoBERTa Logits (Exhaustively)
print("🔄 Reloading MentalRoBERTa for clean 187-sample prediction...")
tok_rob = AutoTokenizer.from_pretrained("mental/mental-roberta-base")
mod_rob = AutoModelForSequenceClassification.from_pretrained("mental/mental-roberta-base", num_labels=9).to("cuda")
# Setting batch size to 1 to prevent sample dropping
trainer_rob_eval = Trainer(model=mod_rob, args=TrainingArguments(output_dir='temp', per_device_eval_batch_size=1, report_to='none'))
rob_val_logits = trainer_rob_eval.predict(DefenseDataset(val_split, tok_rob)).predictions

# 3. Re-extract BERT Logits (Exhaustively)
print("🔄 Reloading MentalBERT for clean 187-sample prediction...")
del mod_rob, trainer_rob_eval
torch.cuda.empty_cache()
gc.collect()

tok_bert = AutoTokenizer.from_pretrained("mental/mental-bert-base-uncased")
mod_bert = AutoModelForSequenceClassification.from_pretrained("mental/mental-bert-base-uncased", num_labels=9).to("cuda")
trainer_bert_eval = Trainer(model=mod_bert, args=TrainingArguments(output_dir='temp', per_device_eval_batch_size=1, report_to='none'))
bert_val_logits = trainer_bert_eval.predict(DefenseDataset(val_split, tok_bert)).predictions

# 4. ENSEMBLE CALCULATIONS (Both should now be exactly 187 samples)
print(f"\n✅ BERT shape: {bert_val_logits.shape}, RoBERTa shape: {rob_val_logits.shape}")

val_final_logits = (bert_val_logits * 0.4) + (rob_val_logits * 0.6)
val_final_preds = np.argmax(val_final_logits, axis=-1)
val_golds = [d['label'] for d in val_split]

print("\n--- 🏆 FINAL HYBRID ENSEMBLE PERFORMANCE ---")
print(f"✅ Accuracy (Leaderboard UI): {accuracy_score(val_golds, val_final_preds):.4f}")
print(f"✅ Macro F1 (Winning Metric): {f1_score(val_golds, val_final_preds, average='macro'):.4f}")
print("\n" + classification_report(val_golds, val_final_preds, zero_division=0))

📊 Running Exhaustive Validation for Ensemble...
🔄 Reloading MentalRoBERTa for clean 187-sample prediction...


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: mental/mental-roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
classifier.out_proj.bias        | MISSING    | 
classifier.dense.bias           | MISSING    | 
classifier.out_proj.weight      | MISSING    | 
classifier.dense.weight         | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to 

🔄 Reloading MentalBERT for clean 187-sample prediction...


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: mental/mental-bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
bert.embeddings.position_ids               | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
bert.pooler.dense.bias                     | MISSING    | 
classifier.weight                          | MISSING    | 
bert.pooler.dense.weight                   | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if 


✅ BERT shape: (187, 9), RoBERTa shape: (187, 9)

--- 🏆 FINAL HYBRID ENSEMBLE PERFORMANCE ---
✅ Accuracy (Leaderboard UI): 0.0160
✅ Macro F1 (Winning Metric): 0.0049

              precision    recall  f1-score   support

           0       0.00      0.00      0.00        30
           1       0.00      0.00      0.00        11
           2       0.00      0.00      0.00         6
           3       0.00      0.00      0.00        10
           4       0.00      0.00      0.00         8
           5       0.00      0.00      0.00         5
           6       0.00      0.00      0.00        17
           7       0.00      0.00      0.00        97
           8       0.02      1.00      0.04         3

    accuracy                           0.02       187
   macro avg       0.00      0.11      0.00       187
weighted avg       0.00      0.02      0.00       187



In [17]:
from sklearn.metrics import classification_report, accuracy_score, f1_score
import numpy as np
import torch
import gc
import os

print("📊 Running Ensemble Validation with TRAINED weights...")

# 1. Clear memory
torch.cuda.empty_cache()
gc.collect()

# 2. Get Trained RoBERTa Logits
# We look for the 'checkpoint-XXXX' folder inside 'rob_f1_stable'
rob_dir = '/kaggle/working/rob_f1_stable'
latest_rob = sorted([os.path.join(rob_dir, d) for d in os.listdir(rob_dir) if 'checkpoint' in d])[-1]
print(f"Loading trained RoBERTa from: {latest_rob}")

mod_rob = AutoModelForSequenceClassification.from_pretrained(latest_rob).to("cuda")
trainer_rob_eval = Trainer(model=mod_rob, args=TrainingArguments(output_dir='temp', per_device_eval_batch_size=1, report_to='none'))
rob_val_logits = trainer_rob_eval.predict(val_ds).predictions

# 3. Get Trained BERT Logits
del mod_rob, trainer_rob_eval
torch.cuda.empty_cache()
gc.collect()

bert_dir = '/kaggle/working/bert_acc'
latest_bert = sorted([os.path.join(bert_dir, d) for d in os.listdir(bert_dir) if 'checkpoint' in d])[-1]
print(f"Loading trained BERT from: {latest_bert}")

mod_bert = AutoModelForSequenceClassification.from_pretrained(latest_bert).to("cuda")
trainer_bert_eval = Trainer(model=mod_bert, args=TrainingArguments(output_dir='temp', per_device_eval_batch_size=1, report_to='none'))
bert_val_logits = trainer_bert_eval.predict(val_ds).predictions

# 4. ENSEMBLE CALCULATIONS
val_final_logits = (bert_val_logits * 0.4) + (rob_val_logits * 0.6)
val_final_preds = np.argmax(val_final_logits, axis=-1)
val_golds = [d['label'] for d in val_split]

print("\n--- 🏆 ACTUAL HYBRID ENSEMBLE PERFORMANCE ---")
print(f"✅ Accuracy: {accuracy_score(val_golds, val_final_preds):.4f}")
print(f"✅ Macro F1: {f1_score(val_golds, val_final_preds, average='macro'):.4f}")
print("\n" + classification_report(val_golds, val_final_preds, zero_division=0))

📊 Running Ensemble Validation with TRAINED weights...
Loading trained RoBERTa from: /kaggle/working/rob_f1_stable/checkpoint-106


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

NameError: name 'val_ds' is not defined

In [18]:
from sklearn.metrics import classification_report, accuracy_score, f1_score
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments
import numpy as np
import torch
import gc
import os

print("📊 Running Ensemble Validation with TRAINED weights...")

# 1. Clear memory
torch.cuda.empty_cache()
gc.collect()

# 2. Get Trained RoBERTa Logits
rob_dir = '/kaggle/working/rob_f1_stable'
latest_rob = sorted([os.path.join(rob_dir, d) for d in os.listdir(rob_dir) if 'checkpoint' in d])[-1]
print(f"Loading trained RoBERTa from: {latest_rob}")

tok_rob = AutoTokenizer.from_pretrained("mental/mental-roberta-base")
mod_rob = AutoModelForSequenceClassification.from_pretrained(latest_rob).to("cuda")
trainer_rob_eval = Trainer(model=mod_rob, args=TrainingArguments(output_dir='temp', per_device_eval_batch_size=1, report_to='none'))

# Using the raw val_split to create the dataset on the fly
rob_val_logits = trainer_rob_eval.predict(DefenseDataset(val_split, tok_rob)).predictions

# 3. Get Trained BERT Logits
del mod_rob, trainer_rob_eval
torch.cuda.empty_cache()
gc.collect()

bert_dir = '/kaggle/working/bert_acc'
latest_bert = sorted([os.path.join(bert_dir, d) for d in os.listdir(bert_dir) if 'checkpoint' in d])[-1]
print(f"Loading trained BERT from: {latest_bert}")

tok_bert = AutoTokenizer.from_pretrained("mental/mental-bert-base-uncased")
mod_bert = AutoModelForSequenceClassification.from_pretrained(latest_bert).to("cuda")
trainer_bert_eval = Trainer(model=mod_bert, args=TrainingArguments(output_dir='temp', per_device_eval_batch_size=1, report_to='none'))

bert_val_logits = trainer_bert_eval.predict(DefenseDataset(val_split, tok_bert)).predictions

# 4. ENSEMBLE CALCULATIONS
val_final_logits = (bert_val_logits * 0.4) + (rob_val_logits * 0.6)
val_final_preds = np.argmax(val_final_logits, axis=-1)
val_golds = [d['label'] for d in val_split]

print("\n--- 🏆 ACTUAL HYBRID ENSEMBLE PERFORMANCE ---")
print(f"✅ Accuracy: {accuracy_score(val_golds, val_final_preds):.4f}")
print(f"✅ Macro F1: {f1_score(val_golds, val_final_preds, average='macro'):.4f}")
print("\n" + classification_report(val_golds, val_final_preds, zero_division=0))

📊 Running Ensemble Validation with TRAINED weights...
Loading trained RoBERTa from: /kaggle/working/rob_f1_stable/checkpoint-106


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Loading trained BERT from: /kaggle/working/bert_acc/checkpoint-265


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]



--- 🏆 ACTUAL HYBRID ENSEMBLE PERFORMANCE ---
✅ Accuracy: 0.6364
✅ Macro F1: 0.2400

              precision    recall  f1-score   support

           0       0.77      0.90      0.83        30
           1       0.17      0.09      0.12        11
           2       0.00      0.00      0.00         6
           3       0.00      0.00      0.00        10
           4       0.67      0.25      0.36         8
           5       0.00      0.00      0.00         5
           6       0.50      0.06      0.11        17
           7       0.63      0.91      0.74        97
           8       0.00      0.00      0.00         3

    accuracy                           0.64       187
   macro avg       0.30      0.25      0.24       187
weighted avg       0.53      0.64      0.55       187

